# 08 - Optics correction

A finite-difference response maps global QF/QD trims to beta beating at four BPMs. Iterated least squares reduces the part of the error that these two knobs can span.

In [ ]:
import Pkg
EXAMPLES_DIR = isfile(joinpath(pwd(), "common.jl")) ? pwd() : joinpath(pwd(), "examples")
Pkg.activate(EXAMPLES_DIR)
using TrackPad, StaticArrays
include(joinpath(EXAMPLES_DIR, "common.jl"))
using .TrackPadExamples

using LinearAlgebra, Statistics

In [ ]:
_, beam = madx_fodo()
ideal = instrumented_fodo(error_kicks=zeros(4))
reference = periodic_twiss(ideal, beam)
target = vcat(bpm_values(ideal, reference.betax), bpm_values(ideal, reference.betay))
errors = (qf=0.025, qd=-0.020)

function beta_vector(trims)
    ring = instrumented_fodo(
        qf=1.414213562373095 + trims[1], qd=-1.414213562373095 + trims[2],
        qf_error=errors.qf, qd_error=errors.qd, error_kicks=zeros(4),
    )
    tw = periodic_twiss(ring, beam)
    vcat(bpm_values(ring, tw.betax), bpm_values(ring, tw.betay))
end

In [ ]:
trims = zeros(2)
before = beta_vector(trims)
for iteration in 1:4
    residual = beta_vector(trims) - target
    h = 1e-5
    response = hcat(((beta_vector(trims + h .* [j == i for j in 1:2]) -
                       beta_vector(trims - h .* [j == i for j in 1:2])) / (2h)
                     for i in 1:2)...)
    trims .-= pinv(response; rtol=1e-10) * residual
end
after = beta_vector(trims)

rms(v) = norm(v) / sqrt(length(v))
(rms_relative_before=rms((before-target)./target),
 rms_relative_after=rms((after-target)./target), family_trims=trims)

Real correction systems usually include phase advance, dispersion, coupling, weights, regularization, and magnet limits. Those policy choices belong in the correction application; TrackPad supplies the model evaluations used to build each response column.